- try 中的 return 会被“挂起”，强制等待 finally 执行完毕。
    - 甚至在 finally 中篡改即将被返回的对象状态。

```python
try:
    return r
finally:
    cleanup()
```

In [3]:
def f():
    try:
        print("try")
        return 1
    finally:
        print("finally")

print(f())

try
finally
1


In [4]:
def h():
    try:
        return 1
    finally:
        return 2

print(h())

2


In [5]:
def k():
    try:
        1 / 0
    finally:
        return 999

print(k())

999


### case 1

In [2]:
import time

class IntelReport:
    def __init__(self, target):
        self.target = target
        self.content = "Blank"
        self.seal_time = None  # 绝密时间戳

    def __repr__(self):
        return f"<报告: 目标={self.target}, 内容='{self.content}', 签章时间={self.seal_time}>"

def extract_intelligence(target_name):
    # 1. 初始化一个可变对象（在内存中开辟了一块具体的空间）
    report = IntelReport(target_name)
    
    try:
        print("特工：正在窃取核心机密...")
        report.content = "发现外星人地下基地"
        
        # 2. 逻辑终点：特工宣告任务完成，提交报告！
        # 此时解释器将 report 所在的【内存引用地址】暂存到返回寄存器中。
        print("特工：情报获取完毕，准备撤退 (执行 Return)！")
        return report  
        
    finally:
        # 3. 强制劫持与状态突变
        # 解释器强制进入 finally。顺着此前暂存的【内存引用地址】，直接修改了该地址内部的数据。
        print("系统门禁：检测到人员撤离，强制对带出的文件盖上不可磨灭的时间戳...")
        time.sleep(1) # 模拟盖章流程耗时
        report.seal_time = time.strftime("%H:%M:%S")

# 外部调用者（总部）接收最终结果
final_document = extract_intelligence("51区")
print("\n总部收到的最终文件：", final_document)

特工：正在窃取核心机密...
特工：情报获取完毕，准备撤退 (执行 Return)！
系统门禁：检测到人员撤离，强制对带出的文件盖上不可磨灭的时间戳...

总部收到的最终文件： <报告: 目标=51区, 内容='发现外星人地下基地', 签章时间=11:56:43>


### case 2

- finally 中释放锁资源

In [1]:
import threading
import time

# 共享资源：全网仅剩 1 件限量商品
inventory = 1  
# 互斥锁：确保同一时刻只有一个线程能操作库存
lock = threading.Lock() 

def purchase_item(user_id):
    global inventory
    
    # 第 1 步：获取锁（进入临界区）
    lock.acquire()  
    print(f"[{user_id}] 成功获取锁，开始处理...")
    
    try:
        # 第 2 步：核心业务逻辑
        if inventory > 0:
            time.sleep(0.1)  # 模拟网络延迟或数据库耗时
            inventory -= 1
            print(f"[{user_id}] 扣减成功！当前库存: {inventory}")
            # 逻辑终点：宣告 return，但控制权将被 finally 强制拦截！
            return True  
        else:
            print(f"[{user_id}] 购买失败，库存不足。")
            return False 
            
    finally:
        # 第 3 步：绝对执行的兜底释放
        lock.release()  
        print(f"[{user_id}] 强制清理完成：锁已释放。\n")

# 模拟两个用户并发抢购
thread_alice = threading.Thread(target=purchase_item, args=("Alice",))
thread_bob = threading.Thread(target=purchase_item, args=("Bob",))

thread_alice.start()
thread_bob.start()

[Alice] 成功获取锁，开始处理...
[Alice] 扣减成功！当前库存: 0
[Alice] 强制清理完成：锁已释放。

[Bob] 成功获取锁，开始处理...
[Bob] 购买失败，库存不足。
[Bob] 强制清理完成：锁已释放。

